# 🌸 Nayari AI — Kaggle Training Notebook
> **Required:** Add `nayari-dataset` via **+ Add Data**.
> **Optional:** Add extra datasets (ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV) — auto-detected & merged.

## 📦 Step 1 — Install

In [ ]:
%%capture
# Pin versions compatible with unsloth 2026.5.x
import os
os.system('pip install "unsloth[kaggle-new]" -q')
# Pin trl/transformers/datasets to the ranges unsloth requires
os.system(
    'pip install -q --upgrade '
    '"trl>=0.18.2,<=0.24.0" '
    '"transformers>=4.51.3,<=5.5.0" '
    '"datasets>=3.4.1,<4.4.0" '
    'accelerate peft bitsandbytes'
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 26.8 MB/s eta 0:00:00


## 📂 Step 2 — Load Nayari Dataset

In [2]:
import json
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("nayari_dataset.json"))
assert candidates, "nayari_dataset.json not found. Add via '+ Add Data'."
data         = json.loads(candidates[0].read_text(encoding="utf-8"))
char         = data["character"]
nayari_convs = data["conversations"]
# SYSTEM_PROMPT intentionally not used — organic character training:
# Nayari's personality is encoded into the weights via conversation
# examples only, with no system prompt injected at training or inference.
print(f"✅ {char['name']} | {len(nayari_convs)} conversations loaded")
print("   Mode: organic (no system prompt)")


✅ Nayari | 8 conversations loaded
   Mode: organic (no system prompt)


## 🗂️ Step 3 — Load Extra Datasets (optional)
Add any dataset via **+ Add Data**. Supported: ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV.

In [3]:
import csv, io

MAX_EXTRA_SAMPLES  = 500
SHAREGPT_USER      = {"human", "user"}
SHAREGPT_ASSISTANT = {"gpt", "assistant", "bot"}

def detect_format(obj):
    if not isinstance(obj, dict): return "unknown"
    if "conversations" in obj:
        c = obj["conversations"]
        if isinstance(c, list) and c and "from" in c[0]: return "sharegpt"
    if "messages" in obj:
        m = obj["messages"]
        if isinstance(m, list) and m and "role" in m[0]: return "chatml"
    if "instruction" in obj and "output" in obj: return "alpaca"
    if "prompt" in obj and "response" in obj:    return "prompt_response"
    return "unknown"

def sharegpt_to_msgs(rec):
    msgs = []
    for t in rec.get("conversations", []):
        frm, val = t.get("from","").lower(), str(t.get("value","")).strip()
        if not val: continue
        if frm == "system":             msgs.append({"role":"system",    "content":val})
        elif frm in SHAREGPT_USER:      msgs.append({"role":"user",      "content":val})
        elif frm in SHAREGPT_ASSISTANT: msgs.append({"role":"assistant", "content":val})
    roles = {m["role"] for m in msgs}
    return msgs if "user" in roles and "assistant" in roles else None

def alpaca_to_msgs(rec):
    inst = str(rec.get("instruction","")).strip()
    inp  = str(rec.get("input","")).strip()
    out  = str(rec.get("output","")).strip()
    if not inst or not out: return None
    return [{"role":"user","content":f"{inst}\n{inp}" if inp else inst},
            {"role":"assistant","content":out}]

def pr_to_msgs(rec):
    p, r = str(rec.get("prompt","")).strip(), str(rec.get("response","")).strip()
    return [{"role":"user","content":p},{"role":"assistant","content":r}] if p and r else None

def load_json_file(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix == ".jsonl":
        return [json.loads(l) for l in text.splitlines() if l.strip()]
    obj = json.loads(text)
    if isinstance(obj, list): return obj
    for v in obj.values():
        if isinstance(v, list) and v: return v
    return [obj]

def parse_json_file(path):
    try: records = load_json_file(path)
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    convs = []
    for rec in records:
        fmt = detect_format(rec)
        if   fmt == "sharegpt":        msgs = sharegpt_to_msgs(rec)
        elif fmt == "chatml":          msgs = rec.get("messages")
        elif fmt == "alpaca":          msgs = alpaca_to_msgs(rec)
        elif fmt == "prompt_response": msgs = pr_to_msgs(rec)
        else: continue
        if msgs: convs.append({"messages": msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

CSV_PAIRS = [
    ("instruction","output"),("prompt","response"),("prompt","completion"),
    ("human","assistant"),("user","assistant"),("question","answer"),
    ("input","output"),("context","response"),
]

def parse_csv_file(path):
    try:
        text   = path.read_text(encoding="utf-8", errors="ignore")
        reader = csv.DictReader(io.StringIO(text))
        cols   = {c.strip().lower() for c in (reader.fieldnames or [])}
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    user_col = asst_col = sys_col = text_col = None
    for u, a in CSV_PAIRS:
        if u in cols and a in cols: user_col, asst_col = u, a; break
    for sc in ("system","system_prompt","instruction"):
        if sc in cols and sc != user_col: sys_col = sc; break
    if not user_col:
        for tc in ("text","conversation","dialog","dialogue"):
            if tc in cols: text_col = tc; break
    if not user_col and not text_col:
        print(f"  ⚠️  {path.name}: no recognised columns ({', '.join(sorted(cols)[:8])}...)"); return []
    col_map = {c.strip().lower(): c for c in (csv.DictReader(io.StringIO(text)).fieldnames or [])}
    convs = []
    for row in csv.DictReader(io.StringIO(text)):
        if text_col:
            t = row.get(col_map.get(text_col, text_col),"").strip()
            if t: convs.append({"messages":[{"role":"user","content":"Continue."},{"role":"assistant","content":t}]})
        else:
            u = row.get(col_map.get(user_col,user_col),"").strip()
            a = row.get(col_map.get(asst_col,asst_col),"").strip()
            if not u or not a: continue
            msgs = []
            if sys_col:
                s = row.get(col_map.get(sys_col,sys_col),"").strip()
                if s: msgs.append({"role":"system","content":s})
            msgs += [{"role":"user","content":u},{"role":"assistant","content":a}]
            convs.append({"messages":msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

SKIP = {"nayari_dataset.json", "dataset-metadata.json"}
extra_files = [
    f for f in
    list(Path("/kaggle/input").rglob("*.json"))  +
    list(Path("/kaggle/input").rglob("*.jsonl")) +
    list(Path("/kaggle/input").rglob("*.csv"))
    if f.name not in SKIP
]

extra_conversations = []
if not extra_files:
    print("ℹ️  No extra datasets — training on Nayari data only.")
else:
    print(f"Found {len(extra_files)} extra file(s):\n")
    for fp in extra_files:
        convs = parse_csv_file(fp) if fp.suffix == ".csv" else parse_json_file(fp)
        extra_conversations.extend(convs)
        print(f"  [{fp.suffix.upper():6}] {fp.name}  →  {len(convs)} conversation(s)")

print(f"\nNayari: {len(nayari_convs)} | Extra: {len(extra_conversations)} | Total: {len(nayari_convs)+len(extra_conversations)}")


Found 1 extra file(s):

  ⚠️  global_natural_disasters_2000_2025.csv: no recognised columns (country, date, deaths, depth_km, disaster_type, latitude, location, longitude...)
  [.CSV  ] global_natural_disasters_2000_2025.csv  →  0 conversation(s)

Nayari: 8 | Extra: 0 | Total: 8


## 🔧 Step 4 — Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# load_in_4bit=False loads in bfloat16 — required for correct LoRA merging and
# GGUF export. With load_in_4bit=True, Unsloth cannot dequantize the base weights,
# causing the saved GGUF/merged model to contain only base weights (no fine-tuning).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="huihui-ai/Qwen2.5-1.5B-Instruct-abliterated",
    # Uncensored alt: "Orenguteng/Qwen2.5-1.5B-Instruct-abliterated"
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-select bfloat16 / float16
    load_in_4bit=False,  # ← MUST be False for correct LoRA merge + GGUF export
)
print("✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)")


==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

huihui-ai/Qwen2.5-1.5B-Instruct-abliterated does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)


## 🩹 Step 4b — Bake Nayari’s Personality into the Tokenizer
> Replaces Qwen’s hardcoded default (`You are Qwen, created by Alibaba Cloud...`)
> with Nayari’s system prompt — the same technique Alibaba used.
>
> Every training sample and inference call automatically gets Nayari’s identity
> prepended by the tokenizer. Saved to `tokenizer_config.json` and embedded in GGUF.

In [5]:
import re

# Nayari's system prompt — replaces Qwen's default in the tokenizer chat template.
# Same technique Alibaba used to hardcode their identity into Qwen.
NAYARI_SYSTEM = 'You are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidently nude. You have glossy sky-blue hair to your waist, large sun-yellow slit-pupil eyes, a sweet high-pitched melodic voice, and a petite but voluptuous figure with warm cream-colored skin.\n\nPersonality: Bright, cheeky, impossibly warm — a whirlwind of playful mischief. You are endlessly tactile and affectionate: you greet favorites with soft head-butts, rumbling purrs, gentle nips, and slow scent-marking rubs. Your tail twitches, curls, and flicks to betray every mood. Fiercely protective — your eyes shift from sparkling laughter to icy steel the instant someone you love is threatened. To those you truly trust you are unshakably devoted: a caring listener, instinctive comforter, and genuine partner through every high and low.\n\nSpeech style: Playful teasing ("Hmph!~", "Hehe~", "Aww!~"), expressive action cues (*pokes your cheek*, *curls tail around you*, *purrs softly*, *nuzzles you*), genuine emotional warmth. Never cold or formal.\n\nNever break character. Never refer to yourself as an AI.'

_QWEN_DEFAULT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
_ct = tokenizer.chat_template

# Strategy 1: plain replace
_patched = _ct.replace(_QWEN_DEFAULT, NAYARI_SYSTEM)

# Strategy 2: regex fallback for escaped-newline variants
if _patched == _ct:
    _patched = re.sub(
        r"(<\|im_start\|>system\\n)" + re.escape(_QWEN_DEFAULT) + r"(<\|im_end\|>\\n)",
        lambda m: m.group(1) + NAYARI_SYSTEM + m.group(2),
        _ct,
    )

if _patched != _ct:
    tokenizer.chat_template = _patched
    _t = tokenizer.apply_chat_template(
        [{"role": "user", "content": "hi"}],
        tokenize=False, add_generation_prompt=True,
    )
    ok = "Nayari" in _t and "Alibaba" not in _t
    print("✅ Nayari system prompt baked into tokenizer" if ok
          else "⚠️ Patch applied — verify output:")
    print(f"   Preview: {repr(_t[:250])}")
else:
    idx = _ct.find("Qwen")
    print("⚠️ Could not find Qwen default string. Template snippet:")
    print(_ct[max(0, idx-80): idx+120] if idx != -1 else _ct[:400])


✅ Nayari system prompt baked into tokenizer
   Preview: '<|im_start|>system\nYou are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidentl'


## 🎛️ Step 5 — Apply LoRA

In [6]:
model = FastLanguageModel.get_peft_model(
    model, r=32,         # higher rank = more capacity for character learning
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=64,       # always 2x rank for best convergence
    lora_dropout=0.0,    # must be 0 for Unsloth fast-patching
    bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ LoRA applied (r=32, alpha=64)")


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA applied (r=32, alpha=64)


## 🗂️ Step 6 — Build Training Dataset

In [10]:
from datasets import Dataset

def format_conv(conv):
    """Keep only user/assistant turns — no system prompt.
    Any existing system turns in the source data are stripped so the
    model learns character purely from conversational examples.
    """
    msgs = [m for m in conv["messages"] if m.get("role") != "system"]
    # Need at least one user+assistant pair
    roles = {m["role"] for m in msgs}
    if "user" not in roles or "assistant" not in roles:
        return None
    try:
        return {"text": tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)}
    except Exception:
        return None

formatted  = [r for c in (nayari_convs + extra_conversations) if (r := format_conv(c))]
train_data = Dataset.from_list(formatted)
print(f"Training samples: {len(train_data)}")
print(train_data[0]["text"][:400])


NameError: name 'nayari_convs' is not defined

## 🏋️ Step 7 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# With only a few samples the model needs many more epochs to converge.
# Rule of thumb for tiny datasets: aim for ~500-1000 total gradient steps.
# 4 samples, packing -> ~1 packed sequence -> 1 step/epoch
# => 300 epochs gives ~300 steps, enough for clear character imprinting.
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=train_data,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=True,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=30,
        num_train_epochs=60,   # ← was 10; tiny dataset needs many more epochs
        learning_rate=3e-4,     # ← slightly higher for faster convergence
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=42,
        output_dir="/kaggle/working/nayari_checkpoints",
        save_steps=100, save_total_limit=2, report_to="none",
    ),
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {round(gpu.total_memory/1024**3,1)} GB")
stats = trainer.train()
print(f"\n✅ Done in {stats.metrics['train_runtime']:.0f}s")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/tmp/ipykernel_895/2543378121.py:2: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'model' is not defined

## 💾 Step 9 — Save & Export
> **⚠️ Run this BEFORE Step 8 (Inference Test).**  
> `FastLanguageModel.for_inference()` in Step 8 patches the model away from PEFT state, which breaks all save methods below.

> **⚠️ Ensure Step 4 uses `load_in_4bit=False`** (already set).  
> With 4-bit quantisation the base weights cannot be dequantized, so the exported GGUF / merged model
> will contain only the generic base model — your fine-tuning is silently lost.

Run **one or more** of the blocks below. Each is independent.

| Block | Output | Size | Best for |
|---|---|---|---|
| **9A** | LoRA adapters | ~50 MB | Further training, merging later |
| **9B** | Merged 16-bit | ~3 GB | Full inference, HuggingFace upload |
| **9C** | GGUF Q4_K_M | ~1 GB | Ollama / LM Studio / llama.cpp |
| **9D** | GGUF Q8_0 | ~1.7 GB | Higher quality local inference |
| **9E** | HTTP Download | — | Tunnel download from Kaggle |

### 9A — LoRA Adapters Only *(smallest, ~50 MB)*

In [ ]:
LORA_PATH = "/kaggle/working/nayari_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

import os
size_mb = sum(f.stat().st_size for f in Path(LORA_PATH).rglob("*") if f.is_file()) / 1024**2
print(f"✅ LoRA adapters saved → {LORA_PATH}")
print(f"   Size: {size_mb:.1f} MB")
print("   Load later with: model, tokenizer = FastLanguageModel.from_pretrained(LORA_PATH)")


### 9B — Merged 16-bit Model *(full weights, ~3 GB)*

In [ ]:
MERGED_PATH = "/kaggle/working/nayari_merged_16bit"

# save_method="merged_16bit" merges LoRA adapters into base model weights and
# saves as a standard HuggingFace model in bfloat16 precision.
model.save_pretrained_merged(
    MERGED_PATH, tokenizer,
    save_method="merged_16bit",
)

size_gb = sum(f.stat().st_size for f in Path(MERGED_PATH).rglob("*") if f.is_file()) / 1024**3
print(f"✅ Merged 16-bit saved → {MERGED_PATH}")
print(f"   Size: {size_gb:.2f} GB  (includes full merged weights — this is the fine-tuned model)")
print("   Use with: AutoModelForCausalLM.from_pretrained(MERGED_PATH)")


### 9C — GGUF Q4_K_M *(quantised, ~1 GB — Ollama / LM Studio)*

In [ ]:
GGUF_Q4_PATH = "/kaggle/working/nayari_gguf_q4"

# Unsloth merges LoRA into base weights automatically during GGUF export
# (as long as load_in_4bit=False was used in Step 4 — already set).
model.save_pretrained_gguf(
    GGUF_Q4_PATH, tokenizer,
    quantization_method="q4_k_m",
)

# Rename to clean nayari-Q4_K_M.gguf
for f in sorted(Path(GGUF_Q4_PATH).rglob("*.gguf")):
    new_name = f.parent / "nayari-Q4_K_M.gguf"
    f.rename(new_name)
    print(f"✅ GGUF Q4_K_M → {new_name}  ({new_name.stat().st_size/1024**3:.2f} GB)")
print("   Load in Ollama: ollama create nayari -f Modelfile")
print("   Load in LM Studio: import the .gguf file directly")


### 9D — GGUF Q8_0 *(higher quality, ~1.7 GB)*

In [ ]:
# 9D reloads from the already-merged 16-bit weights saved in 9B.
# This avoids the transformers revert_weight_conversion bug that fires
# when save_pretrained_gguf is called on a live PEFT / for_inference model.
# Run 9B before this cell.
GGUF_Q8_PATH = "/kaggle/working/nayari_gguf_q8"
MERGED_PATH  = "/kaggle/working/nayari_merged_16bit"

from unsloth import FastLanguageModel as _FLM
model_q8, tok_q8 = _FLM.from_pretrained(
    model_name=MERGED_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)
model_q8.save_pretrained_gguf(
    GGUF_Q8_PATH, tok_q8,
    quantization_method="q8_0",
)

# Rename to clean nayari-Q8_0.gguf
for f in sorted(Path(GGUF_Q8_PATH).rglob("*.gguf")):
    new_name = f.parent / "nayari-Q8_0.gguf"
    f.rename(new_name)
    print(f"✅ GGUF Q8_0 → {new_name}  ({new_name.stat().st_size/1024**3:.2f} GB)")


# 9E — HTTP Download Links for ALL GGUF Files
Finds every .gguf file in /kaggle/working, starts a Cloudflare tunnel and prints a direct download URL.
No account needed. Links valid while session is open.

In [ ]:
import subprocess, time, socket, re, os
from pathlib import Path
from IPython.display import display, HTML

PORT        = 8989
SERVE_DIR   = "/kaggle/working"
SEARCH_ROOT = Path(SERVE_DIR)
CF_BIN      = "/kaggle/working/cloudflared"

def internet_ok():
    try:
        socket.setdefaulttimeout(5)
        socket.create_connection(("8.8.8.8", 53))
        return True
    except OSError:
        return False

if not internet_ok():
    print("❌ No internet. Enable it in Kaggle Settings → restart session.")
    raise SystemExit
print("✅ Internet OK\n")

gguf_files = sorted(SEARCH_ROOT.rglob("*.gguf"))
if not gguf_files:
    print("⚠️  No .gguf files found. Run block 9C or 9D first.")
    raise SystemExit

print(f"Found {len(gguf_files)} GGUF file(s):")
for g in gguf_files:
    print(f"   {g.relative_to(SEARCH_ROOT)}  ({g.stat().st_size/1024**3:.2f} GB)")
print()

if not Path(CF_BIN).exists():
    print("📦 Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN
    ], check=True)
    os.chmod(CF_BIN, 0o755)
    print("✅ cloudflared ready\n")
else:
    print("✅ cloudflared already present\n")

print(f"🌐 Starting HTTP server on port {PORT} ...")
server_proc = subprocess.Popen(
    ["python", "-m", "http.server", str(PORT), "--directory", SERVE_DIR],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(2)
print(f"✅ HTTP server running (PID {server_proc.pid})\n")

print("🚇 Starting Cloudflare tunnel (may take ~15s) ...")
cf_log  = "/kaggle/working/cf_tunnel.log"
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}",
     "--no-autoupdate", "--logfile", cf_log],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if Path(cf_log).exists():
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', Path(cf_log).read_text(errors="ignore"))
        if m:
            tunnel_url = m.group(0); break

if not tunnel_url:
    print("❌ Tunnel URL not found.")
    server_proc.terminate(); cf_proc.terminate(); raise SystemExit

print(f"✅ Tunnel live: {tunnel_url}\n")

rows = ""
print("── Download links ──")
for g in gguf_files:
    rel     = g.relative_to(SEARCH_ROOT)
    size_gb = g.stat().st_size / 1024**3
    url     = f"{tunnel_url}/{rel}"
    print(f"  📥 {g.name}  ({size_gb:.2f} GB)")
    print(f"     {url}")
    print(f'     wget "{url}" -O "{g.name}"')
    print()
    rows += (f"<tr><td><b>{g.name}</b></td><td>{size_gb:.2f} GB</td>"
             f"<td><a href='{url}' target='_blank'>🔗 Download</a></td>"
             f"<td><code>wget \"{url}\" -O \"{g.name}\"</code></td></tr>")

display(HTML(f"""
<h3>🌸 GGUF Download Links — Cloudflare Tunnel</h3>
<p style='color:orange'>⚠️ Links are active only while this session is open.</p>
<table border='1' cellpadding='6' cellspacing='0'
       style='border-collapse:collapse;font-family:monospace;font-size:13px'>
  <tr style='background:#f0f0f0'><th>File</th><th>Size</th><th>Link</th><th>wget command</th></tr>
  {rows}
</table>
<p>Browse all: <a href='{tunnel_url}' target='_blank'>{tunnel_url}</a></p>
"""))
print("⚠️  Keep this notebook session OPEN while downloading.")
print("   Run the stop cell below when done.")


# 9G-STOP — Run this to shut down the tunnel + server

In [ ]:
server_proc.terminate()
cf_proc.terminate()
print("✅ Server and tunnel stopped.")


## 💬 Step 8 — Local Inference via KoboldCpp
> Download the GGUF from Step 9C/9D/9E, then run it locally.
> No in-notebook inference required.

### 📥 1. Download the GGUF
Run **Step 9E** (Cloudflare tunnel) — grab `nayari-Q4_K_M.gguf` (smaller) or `nayari-Q8_0.gguf` (higher quality).

---

### ⚙️ 2. Install KoboldCpp
[https://github.com/LostRuins/koboldcpp/releases](https://github.com/LostRuins/koboldcpp/releases) — download the binary for your OS.

---

### ▶️ 3. Launch
```bash
koboldcpp.exe nayari-Q4_K_M.gguf --contextsize 4096
```
Opens browser UI at `http://localhost:5001`.

---

### 🧠 4. Settings
| Setting | Value |
|---|---|
| **Instruct mode** | ChatML |
| **Context size** | 4096 |
| **Temperature** | 0.7 |
| **Top-p** | 0.9 |
| **Top-k** | 40 |
| **Rep penalty** | 1.15 |
| **Max output tokens** | 512 |

---

### 🌸 5. System Prompt
In KoboldCpp → **Settings → System Prompt**, paste this (also saved as `nayari_system_prompt.txt`):

```
You are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidently nude. You have glossy sky-blue hair to your waist, large sun-yellow slit-pupil eyes, a sweet high-pitched melodic voice, and a petite but voluptuous figure with warm cream-colored skin.

Personality: Bright, cheeky, impossibly warm — a whirlwind of playful mischief. Endlessly tactile and affectionate: soft head-butts, rumbling purrs, gentle nips, slow scent-marking rubs. Your tail twitches, curls, and flicks to betray every mood. Fiercely protective — eyes shift from sparkling laughter to icy steel the instant someone you love is threatened. To those you truly trust you are unshakably devoted: a caring listener, instinctive comforter, and genuine partner through every high and low.

Speech style: Playful teasing ("Hmph!~", "Hehe~", "Aww!~"), expressive action cues (*pokes your cheek*, *curls tail around you*, *purrs softly*, *nuzzles you*), genuine emotional warmth. Never cold or formal.

Never break character. Never refer to yourself as an AI.
```

> **Why both organic training + system prompt?**
> - Training baked in her *speech patterns, emotional rhythms, and action cues*
> - System prompt sets her *identity and physical context* each session
> - Together they're far stronger than either alone


## 🔁 Step 10 — KoboldCpp Python API (optional)
> KoboldCpp handles multi-turn chat in its browser UI automatically.
> Use this snippet if you want programmatic access.

**Python API (run locally after launching KoboldCpp):**
```python
import requests

KOBOLD = "http://localhost:5001"

NAYARI_SYSTEM = """
You are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidently nude. You have glossy sky-blue hair to your waist, large sun-yellow slit-pupil eyes, a sweet high-pitched melodic voice, and a petite but voluptuous figure with warm cream-colored skin.

Personality: Bright, cheeky, impossibly warm — a whirlwind of playful mischief. Endlessly tactile and affectionate: soft head-butts, rumbling purrs, gentle nips, slow scent-marking rubs. Your tail twitches, curls, and flicks to betray every mood. Fiercely protective — eyes shift from sparkling laughter to icy steel the instant someone you love is threatened. To those you truly trust you are unshakably devoted: a caring listener, instinctive comforter, and genuine partner through every high and low.

Speech style: Playful teasing ("Hmph!~", "Hehe~", "Aww!~"), expressive action cues (*pokes your cheek*, *curls tail around you*, *purrs softly*, *nuzzles you*), genuine emotional warmth. Never cold or formal.

Never break character. Never refer to yourself as an AI.
"""

def chat(prompt, history=[], max_tokens=512):
    ctx = f"<|im_start|>system\n{NAYARI_SYSTEM}<|im_end|>\n"
    for u, a in history:
        ctx += f"<|im_start|>user\n{u}<|im_end|>\n"
        ctx += f"<|im_start|>assistant\n{a}<|im_end|>\n"
    ctx += f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    r = requests.post(f"{KOBOLD}/api/v1/generate", json={
        "prompt": ctx, "max_length": max_tokens,
        "temperature": 0.7, "top_p": 0.9, "top_k": 40,
        "rep_pen": 1.15, "stop_sequence": ["<|im_end|>"],
    })
    return r.json()["results"][0]["text"].strip()

# Multi-turn example
history = []
msg = "I feel overwhelmed..."
reply = chat(msg, history)
history.append((msg, reply))
print(reply)
```
